# Sensitivity Plot — Large Text Version
Reproduces the Revenue Sensitivity & Forecast Quality figure from `individual_day_results.csv`.

> **Note on nRMSE (right panel):** The CSV does not store per-day nRMSE values.
> If you have a `nrmse_results.csv` (see Section 2 for format), set `HAVE_NRMSE = True`.
> Otherwise the right panel will be skipped and only the left panel is produced.

In [ ]:
# ── 0. Configuration ─────────────────────────────────────────────────────
RESULTS_CSV  = 'individual_day_results.csv'   # ← adjust path if needed
NRMSE_CSV    = 'nrmse_results.csv'            # ← only needed if HAVE_NRMSE=True
HAVE_NRMSE   = False                          # ← set True if nrmse_results.csv exists

MULTI_SEED   = 999          # used only in the figure title
# Persistence reference values from the original 30-day run (seed=999)
# Replace if you rerun with different settings.
AVG_REV_DAILY   = 0.866     # D-2  (86.6 %)
AVG_REV_WEEKLY  = 0.882     # D-7  (88.2 %)
AVG_NRMSE_DAILY  = 1.30     # approximate from original figure
AVG_NRMSE_WEEKLY = 1.15     # approximate from original figure

# ── Output figure settings ───────────────────────────────────────────────
SAVE_FIGURE = True
FIGURE_PATH = 'sensitivity_summary_large_text.png'
FIGURE_DPI  = 150

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── 2. Load & aggregate results ───────────────────────────────────────────
df = pd.read_csv(RESULTS_CSV)

# ── One-at-a-time: average rev_norm per (var, alpha) ─────────────────────
df_oaat_avg = (
    df[df['sweep'] == 'oaat']
    .groupby(['var', 'alpha'], as_index=False)['rev_norm']
    .mean()
)

# ── All-simultaneous: average rev_norm per alpha ──────────────────────────
df_all_avg = (
    df[df['sweep'] == 'all']
    .groupby('alpha', as_index=False)['rev_norm']
    .mean()
)

# Count sampled days (unique dates at alpha=0, sweep=all)
n_days = df[(df['sweep'] == 'all') & (df['alpha'] == 0.0)]['date'].nunique()
print(f'Days in sample: {n_days}')
print('\nAll-simultaneous averages:')
print(df_all_avg.to_string(index=False))

In [ ]:
# ── 3. (Optional) Load nRMSE results ─────────────────────────────────────
# Expected columns: var, alpha, nrmse   (one row per var × alpha)
# or: var, alpha, nrmse_mean            (already averaged across days)
#
# To generate this CSV from the simulation, add to the main notebook:
#
#   nrmse_rows = []
#   for var in ALL_VAR_KEYS:
#       for alpha in ALPHAS:
#           records = daily_data_oaat[var][alpha]
#           nrmse_key = 'load_early' if var=='load' else ('pv_early' if var=='pv' else var)
#           nrmse_rows.append(dict(var=var, alpha=alpha,
#               nrmse=np.nanmean([r['nrmse'].get(nrmse_key, np.nan) for r in records])))
#   # All-simultaneous nRMSE (mean of all keys)
#   for r_avg in avg_results_all:
#       nrmse_rows.append(dict(var='all', alpha=r_avg['alpha'],
#           nrmse=np.mean(list(r_avg['nrmse'].values()))))
#   pd.DataFrame(nrmse_rows).to_csv('nrmse_results.csv', index=False)

df_nrmse = None
if HAVE_NRMSE:
    df_nrmse = pd.read_csv(NRMSE_CSV)
    print('nRMSE data loaded:')
    print(df_nrmse)
else:
    print('HAVE_NRMSE=False — right panel will be skipped.')

In [ ]:
# ── 4. Colour palette (matches original notebook) ─────────────────────────
C_PV            = '#FFCC00'
C_FCRD_UP_EARLY = '#FF7F0E'
C_FCRD_UP_LATE  = '#D62728'
C_LOAD          = '#1F77B4'
C_FCRD_DN_EARLY = '#00BDF2'
C_FCRD_DN_LATE  = '#2CA02C'
C_SPOT          = '#7F7F7F'
C_DAILY         = '#F42CFF'
C_WEEKLY        = '#9E00A3'

COLORS = {
    'load':          C_LOAD,
    'pv':            C_PV,
    'spot':          C_SPOT,
    'fcrd_up_early': C_FCRD_UP_EARLY,
    'fcrd_dn_early': C_FCRD_DN_EARLY,
    'fcrd_up_late':  C_FCRD_UP_LATE,
    'fcrd_dn_late':  C_FCRD_DN_LATE,
}

VAR_LABELS = {
    'load':          'Load',
    'pv':            'PV generation',
    'spot':          'Spot price',
    'fcrd_up_early': 'FCR-D Up early',
    'fcrd_dn_early': 'FCR-D Dn early',
    'fcrd_up_late':  'FCR-D Up late',
    'fcrd_dn_late':  'FCR-D Dn late',
}

ALL_VAR_KEYS = list(VAR_LABELS.keys())

In [ ]:
# ── 5. Plot ────────────────────────────────────────────────────────────────

# ── Font sizes ─────────────────────────────────────────────────────────────
FS_TITLE_MAIN = 20    # suptitle
FS_TITLE_AX   = 17    # subplot titles
FS_LABEL      = 15    # axis labels
FS_TICK       = 13    # tick labels
FS_LEGEND     = 11    # legend entries
FS_ANNOT      = 11    # annotations (D-2, D-7)

n_panels = 2 if HAVE_NRMSE else 1
fig_w    = 21 if HAVE_NRMSE else 11

fig, axes = plt.subplots(1, n_panels, figsize=(fig_w, 7))
fig.patch.set_facecolor('white')

if n_panels == 1:
    axes = [axes]   # make iterable

fig.suptitle(
    f'Revenue Sensitivity & Forecast Quality — {n_days}-day average'
    f' (seed={MULTI_SEED})',
    fontsize=FS_TITLE_MAIN, fontweight='bold', y=0.98
)

# ── Panel 1: Revenue vs α ──────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('white')

for var in ALL_VAR_KEYS:
    d = df_oaat_avg[df_oaat_avg['var'] == var].sort_values('alpha')
    ax.plot(d['alpha'], d['rev_norm'],
            marker='o', lw=2, ms=5,
            color=COLORS[var], label=VAR_LABELS[var])

ax.plot(df_all_avg['alpha'], df_all_avg['rev_norm'],
        marker='s', lw=3, ms=8, color='#333333',
        label='All simultaneous', zorder=10)

ax.axhline(AVG_REV_DAILY,
           color=C_DAILY, ls='--', lw=2, alpha=0.85,
           label=f'Daily persist. D-2 ({AVG_REV_DAILY:.1%})')
ax.axhline(AVG_REV_WEEKLY,
           color=C_WEEKLY, ls='--', lw=2, alpha=0.85,
           label=f'Weekly persist. D-7 ({AVG_REV_WEEKLY:.1%})')
ax.axhline(1.0, color='gray', ls=':', lw=1)

ax.set_xlabel('α (fraction of σ̂)', fontsize=FS_LABEL)
ax.set_ylabel('Revenue (fraction of perfect)', fontsize=FS_LABEL)
ax.set_title('Revenue vs α  [multi-day avg]',
             fontsize=FS_TITLE_AX, fontweight='bold')
ax.tick_params(axis='both', labelsize=FS_TICK)
ax.legend(fontsize=FS_LEGEND, ncol=2)
ax.grid(False)
ax.spines[['top', 'right']].set_visible(False)

# ── Panel 2: Revenue vs nRMSE (only if nRMSE data available) ──────────────
if HAVE_NRMSE and df_nrmse is not None:
    ax2 = axes[1]
    ax2.set_facecolor('white')

    for var in ALL_VAR_KEYS:
        d_nr = df_nrmse[df_nrmse['var'] == var].sort_values('alpha')
        d_rv = df_oaat_avg[df_oaat_avg['var'] == var].sort_values('alpha')
        # align on alpha
        merged = pd.merge(d_nr, d_rv, on='alpha')
        ax2.plot(merged['nrmse'], merged['rev_norm'],
                 marker='o', lw=2, ms=5,
                 color=COLORS[var], label=VAR_LABELS[var])

    d_all_nr = df_nrmse[df_nrmse['var'] == 'all'].sort_values('alpha')
    d_all_rv = df_all_avg.sort_values('alpha')
    merged_all = pd.merge(d_all_nr, d_all_rv, on='alpha')
    ax2.plot(merged_all['nrmse'], merged_all['rev_norm'],
             marker='s', lw=3, ms=8, color='#333333',
             label='All simultaneous', zorder=10)

    ax2.scatter([AVG_NRMSE_DAILY],  [AVG_REV_DAILY],
                color=C_DAILY, s=200, marker='*',
                zorder=15, edgecolors='black', lw=0.8,
                label='Daily persist.')
    ax2.scatter([AVG_NRMSE_WEEKLY], [AVG_REV_WEEKLY],
                color=C_WEEKLY, s=200, marker='*',
                zorder=15, edgecolors='black', lw=0.8,
                label='Weekly persist.')
    ax2.annotate(
        f'D-2\n({AVG_REV_DAILY:.1%})',
        (AVG_NRMSE_DAILY, AVG_REV_DAILY),
        textcoords='offset points', xytext=(8, -12),
        fontsize=FS_ANNOT, color=C_DAILY
    )
    ax2.annotate(
        f'D-7\n({AVG_REV_WEEKLY:.1%})',
        (AVG_NRMSE_WEEKLY, AVG_REV_WEEKLY),
        textcoords='offset points', xytext=(8, -12),
        fontsize=FS_ANNOT, color=C_WEEKLY
    )

    ax2.axhline(1.0, color='gray', ls=':', lw=1)
    ax2.set_xlabel('nRMSE (forecast quality)', fontsize=FS_LABEL)
    ax2.set_ylabel('Revenue (fraction of perfect)', fontsize=FS_LABEL)
    ax2.set_title('Revenue vs nRMSE  [multi-day avg]',
                  fontsize=FS_TITLE_AX, fontweight='bold')
    ax2.tick_params(axis='both', labelsize=FS_TICK)
    ax2.legend(fontsize=FS_LEGEND, ncol=2)
    ax2.grid(False)
    ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

if SAVE_FIGURE:
    plt.savefig(FIGURE_PATH, dpi=FIGURE_DPI, bbox_inches='tight')
    print(f'Saved: {FIGURE_PATH}')

plt.show()